# Getting Started, Chapter 1 -- Writing a model

`sims-pars` turns a short text description of a stochastic model -- a
**PCore script** -- into a Bayesian network you can sample, intervene on, and
fit to data. This chapter builds one script from scratch, one line at a time.

We use the modern front end, `sims_pars.pcore`, which reports every mistake
with a source location instead of failing silently.

In [ ]:
from sims_pars.pcore import compile_script, check, parse
from sims_pars import sample

## The smallest model

A script is a single `PCore <Name> { ... }` block. Each line inside the braces
declares one **node**. Here is a model with nothing in it yet:

In [ ]:
bn = compile_script('''
PCore SIR {
}
''')
bn

## Value nodes: `name = constant`

The right-hand side of `=` is evaluated once, at compile time. A value node is
just a fixed number (or list).

In [ ]:
bn = compile_script('''
PCore SIR {
    pop = 1000
    i0  = 5
}
''')
sample(bn)

## Distribution nodes: `name ~ dist(...)`

`~` makes the node a random variable. `.sample()` of the distribution is drawn
each time, and the node contributes `logpdf(x)` to the model's log-probability.

The registered distribution tags are `k unif norm lnorm gamma invgamma exp beta
chisq triangle binom pois cat`. Arguments are positional or `name=`.

In [ ]:
bn = compile_script('''
PCore SIR {
    pop = 1000
    beta  ~ unif(1.0, 3.0)
    gamma ~ unif(0.2, 1.0)
}
''')
sample(bn)

## Function nodes: `name = expression`

If the `=` right-hand side references other nodes, it becomes a deterministic
function, recomputed from its parents on every draw. Expressions allow
arithmetic, comparisons, `a if c else b`, and the functions in `MATH_FUNC`
(`exp log sqrt logit ...`). Attribute access, indexing and imports are rejected.

In [ ]:
bn = compile_script('''
PCore SIR {
    pop = 1000
    beta  ~ unif(1.0, 3.0)
    gamma ~ unif(0.2, 1.0)
    r0 = beta / gamma
}
''')
draw = sample(bn)
draw

In [ ]:
# r0 really is beta / gamma every time
abs(draw['r0'] - draw['beta'] / draw['gamma']) < 1e-12

## Exogenous nodes: a bare name

A name that is used but never defined becomes an **exogenous** input: the caller
must supply it at sample time. Below, `prev0` (the initial prevalence) is never
assigned, so it shows up in `bn.Exo` and `sample` requires it in `cond`.

In [ ]:
bn = compile_script('''
PCore SIR {
    pop = 1000
    beta  ~ unif(1.0, 3.0)
    gamma ~ unif(0.2, 1.0)
    r0 = beta / gamma
    cases ~ binom(pop, prev0)
}
''')
bn.Exo

In [ ]:
sample(bn, {'prev0': 0.01})

## Inspecting the network

The compiled object is a `BayesianNetwork`: a directed acyclic graph of nodes.

In [ ]:
print('topological order :', bn.Order)
print('random-variable roots:', bn.RVRoots)   # the free parameters
print('exogenous inputs   :', bn.Exo)
print('leaves             :', bn.Leaves)

## Diagnostics

`check()` never raises -- it returns a list of located diagnostics. Try a script
with a misspelled distribution, a disallowed expression, and a cycle:

In [ ]:
bad = '''
PCore Broken {
    a ~ uniff(0, 1)
    b = a.real + 1
    c = d + 1
    d = c * 2
}
'''
for diag in check(bad):
    print(diag.render(bad))
    print()

`compile_script` raises `DiagnosticError` on any of these; pass `strict=False`
to build what it can and drop the broken nodes instead.

## v2 syntax 1 -- type annotations

`name : type` (one of `float int bool vector simplex`) can precede `~` or `=`.
A constant right-hand side is checked against the declared type at compile time;
a mismatch is a *warning*, the node still builds.

In [ ]:
note = '''
PCore Annotated {
    pop : int   = 1000
    horizon : int = 30.5
    beta : float ~ unif(1.0, 3.0)
}
'''
for d in check(note):
    print(d.render(note))

## v2 syntax 2 -- plates

`for i in lo..hi { ... }` repeats its body for each integer in the (inclusive)
range. A declared name is suffixed (`obs` -> `obs_1, obs_2, ...`); write
`name[i]` in an expression to refer to the matching sibling, and a bare `i`
for the loop value itself.

In [ ]:
bn = compile_script('''
PCore Panel {
    pop = 1000
    p ~ beta(1, 1)
    for w in 1..4 {
        cases ~ binom(pop, p)
        rate = cases[w] / pop
    }
}
''')
print(bn.Order)
sample(bn)

## v2 syntax 3 -- composition with `include`

`include "other.pcore"` splices the nodes of another file's first `PCore` block
into this model. Relative paths resolve against the including file, so pass
`path=` (or run from the file's directory).

In [ ]:
import pathlib, tempfile

work = pathlib.Path(tempfile.mkdtemp())
(work / 'priors.pcore').write_text('''
PCore Priors {
    beta  ~ unif(1.0, 3.0)
    gamma ~ unif(0.2, 1.0)
}
''')

main_src = '''
PCore SIR {
    include "priors.pcore"
    r0 = beta / gamma
}
'''
bn = compile_script(main_src, path=str(work / 'main.pcore'))
print(bn.Order)
sample(bn)

## Round-tripping

A network serialises back to a script or to JSON, and both reload to an
equivalent network.

In [ ]:
print(bn.to_script())

In [ ]:
from sims_pars import bayes_net_from_json
bn2 = bayes_net_from_json(bn.to_json())
bn2.Order == bn.Order

## The model we will carry forward

Chapters 2 and 3 use this SIR model: two uniform priors, a derived `R0`, and an
observed case count.

In [ ]:
SIR_SCRIPT = '''
PCore SIR {
    pop = 1000
    beta  ~ unif(1.0, 3.0)
    gamma ~ unif(0.2, 1.0)
    r0 = beta / gamma
    cases ~ binom(pop, prev0)
}
'''
bn = compile_script(SIR_SCRIPT)
sample(bn, {'prev0': 0.02})

---
**Next:** [Chapter 2 -- Sampling & intervention](GettingStarted02_Sampling and intervention.ipynb)